# Day 3: 複数経路ネットワークと動的利用者均衡 (DUO)

## 学習目標
- 複数経路があるネットワークで、車両が経路をどう学習・更新していくか（`RouteChoice`）を観察する
- `route_choice_principle` の `"homogeneous_DUO"` と `"heterogeneous_DUO"` の違いを比較する
- `duo_update_time` / `duo_update_weight` / `duo_noise` が収束の速さ・安定性にどう影響するかを実験する

## 前提知識
- [`docs/theory.md`](../docs/theory.md) 4節「動的利用者均衡（DUO）」


## DUOのおさらい

DUOは「全員が今分かっている旅行時間情報をもとに一番早そうな経路を選び続けると、
やがてどの経路を選んでも所要時間がほぼ同じになる」という均衡に近づく仕組みでした。
UXsimでは `W.duo_update_time` 秒ごとに、各車両が直近の実績旅行時間を見て経路選択確率
（`route_pref`）を更新します。`duo_update_weight` は「新しい情報をどれだけ重視するか」、
`duo_noise` は「探索的にわざと違う経路を試す確率的揺らぎ」に相当します。


In [ ]:
from uxsim import World

def build_diamond_world(route_choice_principle="homogeneous_DUO", duo_update_time=600, duo_update_weight=0.5, duo_noise=0.01, seed=0):
    """起点origから終点destへ2本の経路(短いが細い経路 / 長いが太い経路)があるダイヤモンド型ネットワーク"""
    W = World(
        name="",
        deltan=5,
        tmax=6000,
        duo_update_time=duo_update_time,
        duo_update_weight=duo_update_weight,
        duo_noise=duo_noise,
        route_choice_principle=route_choice_principle,
        print_mode=0, save_mode=1, show_mode=0,
        random_seed=seed,
    )
    W.addNode("orig", 0, 0)
    W.addNode("mid1", 1, 1)   # 経由地1: 短いが車線が細く詰まりやすい経路
    W.addNode("mid2", 1, -1)  # 経由地2: 遠回りだが余裕がある経路
    W.addNode("dest", 2, 0)

    W.addLink("o_mid1", "orig", "mid1", length=1000, free_flow_speed=20, number_of_lanes=1)
    W.addLink("mid1_d", "mid1", "dest", length=1000, free_flow_speed=20, number_of_lanes=1)
    W.addLink("o_mid2", "orig", "mid2", length=1800, free_flow_speed=20, number_of_lanes=2)
    W.addLink("mid2_d", "mid2", "dest", length=1800, free_flow_speed=20, number_of_lanes=2)

    W.adddemand("orig", "dest", t_start=0, t_end=4000, flow=0.6)
    return W

W = build_diamond_world()
W.exec_simulation()
W.analyzer.print_simple_stats()


In [ ]:
# 各経路(リンクの組)の利用台数の時系列を見て、均衡に近づいていく様子を確認する
df = W.analyzer.link_traffic_state_to_pandas()
for link_name in ["o_mid1", "o_mid2"]:
    total = df[df["link"] == link_name]["q"].sum()
    print(f"{link_name} を通過した総流動量の目安(qの合計): {total:.1f}")


## Part B: 演習

1. `duo_update_time` を `60` と `600` で比較し、`W.analyzer.print_simple_stats()` の
   `average travel time` がどう違うか、収束の速さがどう変わるかを確認してください。
2. `route_choice_principle` を `"heterogeneous_DUO"` に切り替えて同じ実験を行い、
   結果の違いを考察してください（`heterogeneous_DUO` は車両ごとに異なる `route_pref` を
   持てる設定です）。
3. `duo_noise` を `0.01` → `0.2` に上げると、経路選択の揺らぎ（探索的な逸脱）がどう変わるか
   観察してください。


In [ ]:
# TODO: duo_update_timeを変えて比較する
# W_fast = build_diamond_world(duo_update_time=60)
# W_slow = build_diamond_world(duo_update_time=600)
# それぞれ exec_simulation() して average_travel_time を比較する


In [ ]:
# TODO: route_choice_principleをheterogeneous_DUOに変えて比較する
# W_hetero = build_diamond_world(route_choice_principle="heterogeneous_DUO")


## Part C: 考察

- 今回のDUOは「各車両が自分の旅行時間を最小化しようとする」自己中心的な均衡でした。
  もし交通管制センターが「ネットワーク全体の総旅行時間」を最小化するように経路を指示できたら、
  DUOの結果と同じになるでしょうか？ 違うとしたら、どちらが「短い経路に台数を詰め込みすぎる」
  でしょうか？（→ Day 5 でこの問いを実際に解きます）
